# 02 EDA and preprocessing
CPU runtime. EDA on the raw subset, then the same JPEG pipeline for every image.

Refs: Grommelt et al. 2024, "Fake or JPEG?" (arXiv 2403.17608)

In [ ]:
# setup (same in every notebook)
import os, sys, subprocess

try:  # autoreload breaks on some Colab Python versions
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload unavailable, restart runtime after pulling changes")

REPO_URL = "https://github.com/Devanshu1503/CV_project_genimage-detection.git"
REPO_DIR = "/content/CV_project_genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run from notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()  # config dict + paths

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

from src import eda_utils as eu
from src import image_utils as iu
from src import manifest as mf
from src.config import generator_names, get_logger

eu.set_style()
pd.set_option("display.width", 200, "display.max_columns", 30)
log = get_logger("02_eda_and_preprocess", P.logs)
SEED = cfg["project"]["seed"]
NAMES = generator_names(cfg)
GEN_ORDER = [NAMES[g["folder"]] for g in cfg["generators"]]
E = cfg["eda"]
FIG = P.figures


def save(fig, name):
    eu.savefig(fig, f"02_{name}", FIG)
    plt.show()


df = mf.read_csv(P.manifests / "subset_index.csv")
# real/fake names for plots
df["class_name"] = df["label"].map({"nature": "real", "ai": "fake"})
df["gen"] = pd.Categorical(df["generator_name"], GEN_ORDER, ordered=True)
# "Real" or the generator name
df["source"] = np.where(df["label"] == "nature", "Real", df["generator_name"])
SOURCE_ORDER = ["Real"] + GEN_ORDER
df["source"] = pd.Categorical(df["source"], SOURCE_ORDER, ordered=True)
df["is_jpeg"] = (df["real_format"] == "JPEG").astype(int)
df["pixels"] = df["width"] * df["height"]
df["aspect"] = df["width"] / df["height"]
df["kb"] = df["file_bytes"] / 1024
print(f"{len(df):,} images loaded from subset_index.csv")

# Part 1: EDA on the raw subset
## 1. Schema, missing values, corrupt files

In [ ]:
schema = pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(),
                       "missing": df.isna().sum(), "missing_%": (df.isna().mean() * 100).round(2),
                       "n_unique": df.nunique()})
display(schema)
display(df.describe().T)

Missing `jpeg_quality` is expected for PNGs.

In [ ]:
print("Missing jpeg_quality by real format:")
display(df.groupby("real_format", dropna=False)["jpeg_quality"].apply(lambda s: s.isna().sum()).rename("missing"))
dropped = pd.read_csv(P.manifests / "dropped_files.csv") if (P.manifests / "dropped_files.csv").exists() else pd.DataFrame()
print(f"Files dropped in notebook 01: {len(dropped)}")
if len(dropped):
    display(dropped["drop_reason"].value_counts())
print("Unreadable files remaining in the index:", int((df["readable"] != True).sum()))

## 2. Record counts

In [ ]:
counts = df.groupby(["gen", "role", "class_name"], observed=True).size().rename("n").reset_index()
display(counts.pivot_table(index=["gen", "role"], columns="class_name", values="n", observed=True))
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=counts, x="gen", y="n", hue=counts["role"] + " / " + counts["class_name"], ax=ax)
ax.set(xlabel="", ylabel="images", title="Images per generator, role and label")
ax.legend(fontsize=8)
save(fig, "record_counts")

## 3. Format bias

In [ ]:
fmt_label = pd.crosstab(df["real_format"].fillna("?"), df["class_name"], margins=True)
fmt_gen = pd.crosstab(df["source"], df["real_format"].fillna("?"))
display(fmt_label)
display(fmt_gen)
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(fmt_gen.div(fmt_gen.sum(1), axis=0), annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set(title="Share of each file format per source", xlabel="real file format", ylabel="")
save(fig, "format_heatmap")

jq = df.dropna(subset=["jpeg_quality"])
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.histplot(data=jq, x="jpeg_quality", hue="class_name", palette=eu.LABEL_PALETTE, discrete=True, ax=ax)
ax.set(title="Estimated JPEG quality (JPEG files only)", xlabel="IJG quality")
save(fig, "jpeg_quality_hist")
print("JPEG quality summary per label:")
display(jq.groupby("class_name")["jpeg_quality"].describe())
print("Share of JPEG quality estimates that exactly match a standard libjpeg table:",
      round(jq["jpeg_q_exact"].astype(bool).mean(), 3))

## 4. Size bias

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(data=df.sample(min(len(df), 6000), random_state=SEED), x="width", y="height",
                hue="source", style="class_name", alpha=0.5, s=14, ax=ax)
ax.set(xscale="log", yscale="log", title="Image size (log scale): real vs each generator")
ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
save(fig, "size_scatter")

print("Most common sizes per source:")
display(df.groupby("source", observed=True)[["width", "height"]].apply(
    lambda g: (g["width"].astype(int).astype(str) + "x" + g["height"].astype(int).astype(str))
    .value_counts().head(3).to_dict()).rename("top sizes").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x="pixels", hue="source", log_scale=True, element="step", fill=False, ax=axes[0])
axes[0].set(title="Number of pixels per image (log)")
sns.violinplot(data=df, x="source", y="kb", hue="class_name", palette=eu.LABEL_PALETTE, log_scale=True,
               cut=0, ax=axes[1])
axes[1].set(title="File size (KB, log) per source", xlabel="")
axes[1].tick_params(axis="x", rotation=45)
save(fig, "size_hist_filesize_violin")

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.histplot(data=df, x="aspect", hue="class_name", palette=eu.LABEL_PALETTE, bins=60, ax=ax)
ax.set(title="Aspect ratio (width / height)", xlabel="aspect ratio")
save(fig, "aspect_ratio")

## 5. Pixel statistics
On a sample per generator and label.

In [ ]:
pix_sample = eu.sample_groups(df, E["pixel_stats_per_group"], SEED, ("generator", "label"))
pix, hists = eu.compute_pixel_stats(pix_sample, P.project_root, "raw_relpath", E["pixel_stats_max_side"])
pix["source"] = pd.Categorical(np.where(pix["label"] == "nature", "Real", pix["generator_name"]), SOURCE_ORDER, ordered=True)
pix.to_csv(P.results / "eda_pixel_stats.csv", index=False)

srcs = [s for s in SOURCE_ORDER if (pix["source"] == s).any()]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
x = np.linspace(0, 255, hists.shape[2])
for c, (ax, cname) in enumerate(zip(axes, ["red", "green", "blue"])):
    for s in srcs:
        m = (pix["source"] == s).to_numpy()
        ax.plot(x, np.nanmean(hists[m, c, :], axis=0), label=s, lw=2.5 if s == "Real" else 1.2,
                color="k" if s == "Real" else None)
    ax.set(title=f"{cname} channel", xlabel="intensity")
axes[0].set_ylabel("density")
axes[2].legend(fontsize=7)
save(fig, "channel_histograms")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["brightness", "contrast", "saturation"]):
    sns.violinplot(data=pix, x="source", y=col, cut=0, ax=ax, order=srcs)
    ax.tick_params(axis="x", rotation=45)
    ax.set(xlabel="", title=col)
save(fig, "brightness_contrast_saturation")

In [ ]:
# Q-Q: real vs fake brightness
real_b = pix.loc[pix["label"] == "nature", "brightness"].dropna().to_numpy()
fake_b = pix.loc[pix["label"] == "ai", "brightness"].dropna().to_numpy()
q = np.linspace(0.01, 0.99, 99)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(np.quantile(real_b, q), np.quantile(fake_b, q), s=10)
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set(xlabel="real brightness quantiles", ylabel="fake brightness quantiles", title="Q-Q: real vs fake brightness")
stats.probplot(real_b, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q: real brightness vs normal")
save(fig, "qq_brightness")
ks = stats.ks_2samp(real_b, fake_b)
print(f"Kolmogorov-Smirnov test real vs fake brightness: D={ks.statistic:.3f}, p={ks.pvalue:.2e}")

print("Grayscale images per source (whole subset):")
display(df.groupby("source", observed=True)["is_gray"].apply(lambda s: s.astype(str).eq("True").sum()).rename("n_gray"))

## 6. Sample grids

In [ ]:
for s in SOURCE_ORDER:
    g = df.loc[df["source"] == s]
    if g.empty:
        continue
    g = g.sample(min(len(g), E["grid_n"]), random_state=SEED)
    caps = [f"{int(w)}x{int(h)} {f}" for w, h, f in zip(g["width"], g["height"], g["real_format"])]
    fig = eu.plot_image_grid([P.project_root / r for r in g["raw_relpath"]], caps, ncols=E["grid_n"], title=s)
    save(fig, f"grid_{s.replace('.', '')}")

## 7. Frequency analysis
Mean log FFT of a center crop (no resize), plus the difference vs real.

Refs: Frank et al. 2020 (arXiv 2003.08685)

In [ ]:
fft_sample = eu.sample_groups(df, E["fft_per_group"], SEED, ("source",))
spectra, fft_counts = eu.average_spectra(fft_sample, P.project_root, "raw_relpath", "source", E["fft_crop"])
display(pd.DataFrame(fft_counts).T)
spectra = {k: spectra[k] for k in SOURCE_ORDER if k in spectra}
fig = eu.plot_spectra(spectra, reference="Real", title=f"Mean log-|FFT| of {E['fft_crop']}px grayscale center crops (raw)")
save(fig, "fft_spectra_raw")

## 8. Class balance

In [ ]:
cov = df.groupby(["gen", "role", "class_name"], observed=True).agg(
    n=("sample_id", "size"), with_class=("imagenet_class", lambda s: s.notna().sum()),
    n_classes=("imagenet_class", "nunique"),
    max_per_class=("imagenet_class", lambda s: s.value_counts().max() if s.notna().any() else 0))
display(cov)
train_real = df.loc[(df["role"] == "train") & (df["label"] == "nature"), "imagenet_class"].dropna()
if len(train_real):
    fig, ax = plt.subplots(figsize=(8, 3))
    train_real.value_counts().plot.hist(bins=30, ax=ax)
    ax.set(title="Training real images: images per ImageNet class", xlabel="images in the class", ylabel="classes")
    save(fig, "class_balance_train_real")

## 9. Outliers

In [ ]:
df["out_tiny"] = df[["width", "height"]].min(axis=1) < 128
df["out_aspect"] = (df["aspect"] > 2) | (df["aspect"] < 0.5)
df["out_gray"] = df["is_gray"].astype(str).eq("True")
df["out_filesize"] = df.groupby("source", observed=True)["file_bytes"].transform(eu.iqr_outlier_mask)
pairs = pd.read_csv(P.manifests / "duplicate_pairs.csv") if (P.manifests / "duplicate_pairs.csv").exists() else pd.DataFrame()
dup_ids = set(pairs.get("a_sample_id", [])) | set(pairs.get("b_sample_id", []))
df["out_near_dup"] = df["sample_id"].isin(dup_ids)
out_cols = ["out_tiny", "out_aspect", "out_gray", "out_filesize", "out_near_dup"]
display(df.groupby("source", observed=True)[out_cols].sum())

for col in out_cols:
    g = df.loc[df[col]]
    if g.empty:
        continue
    g = g.sample(min(len(g), 8), random_state=SEED)
    caps = [f"{s}\n{int(w)}x{int(h)} {k:.0f}KB" for s, w, h, k in zip(g["source"], g["width"], g["height"], g["kb"])]
    fig = eu.plot_image_grid([P.project_root / r for r in g["raw_relpath"]], caps, ncols=8, title=f"Outliers: {col}")
    save(fig, f"outliers_{col}")

## 10. Shortcut test (metadata only)
If metadata alone predicts real vs fake, the data has a shortcut.

In [ ]:
META_FEATS = ["is_jpeg", "width", "height", "file_bytes", "jpeg_quality"]
shortcut_raw = {}
for name, sub in {"train_pool_SD1.4": df.loc[df["role"] == "train"], "all_images": df}.items():
    table, cms = eu.metadata_shortcut_test(sub, META_FEATS, "y", SEED, E["cv_folds"])
    table.insert(0, "data", name)
    shortcut_raw[name] = (table, cms)
    display(table.round(4))
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, (mname, cm) in zip(axes, shortcut_raw["all_images"][1].items()):
    eu.plot_confusion(cm, f"RAW metadata: {mname}", ax)
save(fig, "shortcut_confusion_raw")

from sklearn.tree import DecisionTreeClassifier, export_text
tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(df[META_FEATS].fillna(-1), df["y"])
print("The decision tree's rules (how the shortcut works):\n" + export_text(tree, feature_names=META_FEATS))

raw_metrics = pd.concat([t for t, _ in shortcut_raw.values()])
raw_metrics.to_json(P.results / "shortcut_test_raw.json", orient="records", indent=2)

# Part 2: preprocessing
## 11. Same pipeline for every image
RGB, center square crop, 224 bicubic, JPEG q96.

In [ ]:
PP = cfg["preprocess"]
df["processed_relpath"] = ("processed/" + df["generator"] + "/" + df["role"] + "/" + df["label"] + "/"
                           + df["filename"].str.rsplit(".", n=1).str[0] + ".jpg")
assert df["processed_relpath"].is_unique, "two images would map to the same processed file"
pairs_io = list(zip([P.project_root / r for r in df["raw_relpath"]], [P.project_root / r for r in df["processed_relpath"]]))
todo = sum(not d.exists() for _, d in pairs_io)
print(f"{len(pairs_io) - todo:,} already processed, {todo:,} to do")
errors = iu.process_many(pairs_io, n_threads=PP["n_threads"], size=PP["image_size"], quality=PP["jpeg_quality"],
                         subsampling=PP["jpeg_subsampling"], resample=PP["resample"])
df["process_error"] = errors
n_err = df["process_error"].notna().sum()
print(f"Processing errors: {n_err}")
if n_err:
    log.error(f"{n_err} preprocessing errors")
    display(df.loc[df["process_error"].notna(), ["sample_id", "process_error"]].head(20))

## 12. `processed_index.csv`

In [ ]:
ok = df.loc[df["process_error"].isna()].copy()
pmeta = pd.DataFrame(iu.metadata_many([P.project_root / r for r in ok["processed_relpath"]],
                                      n_threads=PP["n_threads"], with_hashes=False))
pmeta = pmeta[["real_format", "width", "height", "file_bytes", "jpeg_quality"]].add_prefix("p_")
ok = pd.concat([ok.reset_index(drop=True), pmeta], axis=1)
base_cols = list(mf.read_csv(P.manifests / "subset_index.csv", nrows=1).columns)
processed_index = ok[base_cols + ["processed_relpath"] + list(pmeta.columns)]
processed_index.to_csv(P.manifests / "processed_index.csv", index=False)
print(f"Saved processed_index.csv: {len(processed_index):,} rows")
display(processed_index[list(pmeta.columns)].describe(include="all").T)

## 13. Train / val / test split
SD1.4 pool split 80/20, every generator's sample is a test set.

In [ ]:
from sklearn.model_selection import train_test_split
pool = processed_index.loc[processed_index["role"] == "train"]
tr, va = train_test_split(pool["sample_id"], test_size=cfg["split"]["val_fraction"],
                          stratify=pool["y"], random_state=SEED)
splits = pd.concat([
    pd.DataFrame({"sample_id": tr, "split_set": "train"}),
    pd.DataFrame({"sample_id": va, "split_set": "val"}),
    pd.DataFrame({"sample_id": processed_index.loc[processed_index["role"] == "test", "sample_id"], "split_set": "test"}),
], ignore_index=True)
splits.to_csv(P.manifests / "splits.csv", index=False)
chk = processed_index.merge(splits, on="sample_id")
display(chk.groupby(["split_set", "generator_name", "label"]).size().unstack("label"))

## 14. Does the shortcut go away?
Same test on the processed files, with and without file size.

In [ ]:
pdf = processed_index.copy()
pdf["source"] = pd.Categorical(np.where(pdf["label"] == "nature", "Real", pdf["generator_name"]), SOURCE_ORDER, ordered=True)
# same features, measured on processed files
pdf_m = pd.DataFrame({"y": pdf["y"], "role": pdf["role"],
                      "is_jpeg": (pdf["p_real_format"] == "JPEG").astype(int),
                      "width": pdf["p_width"], "height": pdf["p_height"],
                      "file_bytes": pdf["p_file_bytes"], "jpeg_quality": pdf["p_jpeg_quality"]})
rows = []
for feat_name, feats in {"format+size+quality+bytes": META_FEATS,
                         "format+size+quality (no bytes)": ["is_jpeg", "width", "height", "jpeg_quality"]}.items():
    for data_name, sub_raw, sub_p in [("train_pool_SD1.4", df.loc[df["role"] == "train"], pdf_m.loc[pdf_m["role"] == "train"]),
                                      ("all_images", df, pdf_m)]:
        t_raw, _ = eu.metadata_shortcut_test(sub_raw, feats, "y", SEED, E["cv_folds"])
        t_p, cms_p = eu.metadata_shortcut_test(sub_p, feats, "y", SEED, E["cv_folds"])
        for stage, t in [("raw", t_raw), ("processed", t_p)]:
            t = t.assign(stage=stage, data=data_name, feature_set=feat_name)
            rows.append(t)
compare = pd.concat(rows, ignore_index=True)
compare.to_json(P.results / "shortcut_test_before_after.json", orient="records", indent=2)
display(compare.pivot_table(index=["data", "feature_set", "model"], columns="stage",
                            values=["accuracy_mean", "roc_auc_mean"]).round(3))

In [ ]:
# format/size plots again, processed
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fmt_p = pd.crosstab(pdf["source"], pdf["p_real_format"])
sns.heatmap(fmt_p.div(fmt_p.sum(1), axis=0), annot=True, fmt=".2f", cmap="Blues", ax=axes[0])
axes[0].set(title="Processed: file format per source", ylabel="")
sns.stripplot(data=pdf.sample(min(len(pdf), 3000), random_state=SEED), x="p_width", y="source", hue="label",
              palette=eu.LABEL_PALETTE, ax=axes[1], size=3, jitter=0.3)
axes[1].set(title="Processed: width per source", xlabel="width (px)", ylabel="")
sns.violinplot(data=pdf, x="source", y=pdf["p_file_bytes"] / 1024, hue="label", palette=eu.LABEL_PALETTE, cut=0, ax=axes[2])
axes[2].set(title="Processed: file size (KB)", xlabel="", ylabel="KB")
axes[2].tick_params(axis="x", rotation=45)
save(fig, "processed_format_size")

In [ ]:
# spectra after preprocessing
pfft_sample = eu.sample_groups(pdf, E["fft_per_group"], SEED, ("source",))
pspectra, _ = eu.average_spectra(pfft_sample, P.project_root, "processed_relpath", "source", E["fft_crop"])
pspectra = {k: pspectra[k] for k in SOURCE_ORDER if k in pspectra}
fig = eu.plot_spectra(pspectra, reference="Real", title="Mean log-|FFT| after preprocessing")
save(fig, "fft_spectra_processed")

## 15. Original vs processed

In [ ]:
ex = eu.sample_groups(processed_index, 3, SEED, ("label",)).head(5)
paths, caps = [], []
for _, r in ex.iterrows():
    paths += [P.project_root / r["raw_relpath"], P.project_root / r["processed_relpath"]]
    caps += [f"ORIGINAL {r['generator_name']}/{r['label']}\n{int(r['width'])}x{int(r['height'])} {r['real_format']}",
             f"PROCESSED\n{int(r['p_width'])}x{int(r['p_height'])} {r['p_real_format']} q{r['p_jpeg_quality']:.0f}"]
fig = eu.plot_image_grid(paths, caps, ncols=2, ax_size=3.2, title="Original (left) vs processed (right)")
save(fig, "original_vs_processed")

## 16. Limitations
- Real images get JPEG-compressed twice, fakes only once.
- BigGAN (128px) gets upscaled, Midjourney (1024px) heavily downscaled.
- Center crop drops the edges of non-square images.
- File size still depends on content (see 14).

## Findings
- Schema / missing values / corrupt files: 18,000 rows, 0 corrupt, 0 dropped. Only expected gaps: `jpeg_quality` is empty for the 9,000 PNGs, `imagenet_class` for 5,000 files (GLIDE/VQDM fakes and val reals).
- Format bias: all 9,000 real images are JPEG (median quality 96, mean 92.7), all 9,000 fakes are PNG.
- Size bias: each generator has one fixed size (SD1.4/SD1.5/Wukong 512, Midjourney 1024, ADM/GLIDE/VQDM 256, BigGAN 128). Real sizes vary, mostly 500x375.
- Pixel statistics: brightness differs only a little, KS D = 0.079 (significant with this many images, but small).
- Frequency spectra: BigGAN shows a grid of FFT peaks (GAN upsampling artifacts), GLIDE is smoother than real, SD1.4/SD1.5/Wukong have extra high-frequency energy that survives preprocessing.
- Class balance and outliers: train is 1000 classes x 5 per label. Outliers: 149 tiny reals, 170 grayscale reals, 17 grayscale Midjourney fakes, all kept. Splits: train 4000 + 4000, val 1000 + 1000, test 500 + 500 x 8 generators.
- Shortcut test RAW: a depth-3 tree only needs `is_jpeg` and gets 100% accuracy (AUROC 1.0). Logistic regression gets 99.99%.
- Shortcut test PROCESSED: drops to 50% (chance). With file size it is 52-56%, because JPEG size depends on image content.

## Run before closing the tab
Uploads pending Drive writes, otherwise the next session may not see them.

In [ ]:
from src.colab_setup import flush_drive
flush_drive()